# Initial profiling: huggingface_spotify.csv

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("huggingface_spotify.csv", index_col=0)
df.head()

,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.4610,1,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.715,87.917,4,acoustic
1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,0.1660,1,-17.235,1,0.0763,0.9240,0.000006,0.1010,0.267,77.489,4,acoustic
2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,0.3590,0,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.120,76.332,4,acoustic
3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,False,0.266,0.0596,0,-18.515,1,0.0363,0.9050,0.000071,0.1320,0.143,181.740,3,acoustic
4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,False,0.618,0.4430,2,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.167,119.949,4,acoustic


## 1. Row counts and structure

In [15]:
print(f"Rows: {len(df):,}")
print(f"Columns: {df.shape[1]}")
print(f"Unique track_ids: {df['track_id'].nunique():,}")
print(f"Unique genres: {df['track_genre'].nunique()}")
print()
df.info()

Rows: 114,000
Columns: 20
Unique track_ids: 89,741
Unique genres: 114

<class 'pandas.core.frame.DataFrame'>
Index: 114000 entries, 0 to 113999
Data columns (total 20 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   track_id          114000 non-null  object 
 1   artists           113999 non-null  object 
 2   album_name        113999 non-null  object 
 3   track_name        113999 non-null  object 
 4   popularity        114000 non-null  int64  
 5   duration_ms       114000 non-null  int64  
 6   explicit          114000 non-null  bool   
 7   danceability      114000 non-null  float64
 8   energy            114000 non-null  float64
 9   key               114000 non-null  int64  
 10  loudness          114000 non-null  float64
 11  mode              114000 non-null  int64  
 12  speechiness       114000 non-null  float64
 13  acousticness      114000 non-null  float64
 14  instrumentalness  114000 non-null  float64
 15  li

In [ ]:
genre_counts = df["track_genre"].value_counts()
print(genre_counts.describe())
genre_counts.head(10)

count     114.0
mean     1000.0
std         0.0
min      1000.0
25%      1000.0
50%      1000.0
75%      1000.0
max      1000.0
Name: count, dtype: float64


track_genre
acoustic             1000
punk-rock            1000
progressive-house    1000
power-pop            1000
pop                  1000
pop-film             1000
piano                1000
party                1000
pagode               1000
opera                1000
Name: count, dtype: int64

## 2. Duplicate track_ids (tracks listed under more than one genre)

In [ ]:

print(f"Exact duplicate rows: {df.duplicated().sum():,}")

id_counts = df["track_id"].value_counts()
dup_ids = id_counts[id_counts > 1]
print(f"track_ids appearing more than once: {len(dup_ids):,}")
print(f"Rows involved in duplicated track_ids: {dup_ids.sum():,}")
print(f"Extra rows beyond first occurrence: {(dup_ids - 1).sum():,}")

genres_per_id = df.groupby("track_id")["track_genre"].nunique()
multi_genre = genres_per_id[genres_per_id > 1]
print(f"track_ids listed under >1 genre: {len(multi_genre):,}")

same_genre_dups = df.duplicated(subset=["track_id", "track_genre"]).sum()
print(f"Rows repeating the same track_id + genre: {same_genre_dups:,}")

Exact duplicate rows: 450
track_ids appearing more than once: 16,641
Rows involved in duplicated track_ids: 40,900
Extra rows beyond first occurrence: 24,259
track_ids listed under >1 genre: 16,299
Rows repeating the same track_id + genre: 450


In [ ]:
print(multi_genre.value_counts().sort_index().rename("num_tracks"))

top_id = multi_genre.idxmax()
df.loc[df["track_id"] == top_id, ["track_id", "track_name", "artists", "track_genre", "popularity"]]

track_genre
2    11424
3     2955
4     1361
5      431
6      104
7       21
8        2
9        1
Name: num_tracks, dtype: int64


,track_id,track_name,artists,track_genre,popularity
8315,6S3JlDAGk3uu3NtZbPnuhS,Baby Blue - Remastered 2010,Badfinger,blues,67
19759,6S3JlDAGk3uu3NtZbPnuhS,Baby Blue - Remastered 2010,Badfinger,country,67
34728,6S3JlDAGk3uu3NtZbPnuhS,Baby Blue - Remastered 2010,Badfinger,folk,67
62226,6S3JlDAGk3uu3NtZbPnuhS,Baby Blue - Remastered 2010,Badfinger,j-pop,67
63087,6S3JlDAGk3uu3NtZbPnuhS,Baby Blue - Remastered 2010,Badfinger,j-rock,67
82064,6S3JlDAGk3uu3NtZbPnuhS,Baby Blue - Remastered 2010,Badfinger,power-pop,67
84129,6S3JlDAGk3uu3NtZbPnuhS,Baby Blue - Remastered 2010,Badfinger,psych-rock,67
99727,6S3JlDAGk3uu3NtZbPnuhS,Baby Blue - Remastered 2010,Badfinger,singer-songwriter,67
102732,6S3JlDAGk3uu3NtZbPnuhS,Baby Blue - Remastered 2010,Badfinger,songwriter,67


In [ ]:
feature_cols = [c for c in df.columns if c != "track_genre"]
dup_rows = df[df["track_id"].isin(dup_ids.index)]
inconsistent = (dup_rows.groupby("track_id")[feature_cols].nunique() > 1).sum()
print("Columns that differ within the same track_id (count of track_ids):")
print(inconsistent[inconsistent > 0])

Columns that differ within the same track_id (count of track_ids):
popularity    720
dtype: int64


## 3. Missing values

In [20]:
missing = df.isna().sum()
missing_pct = (missing / len(df) * 100).round(4)
missing_report = pd.DataFrame({"missing": missing, "pct": missing_pct})
print(f"Total missing cells: {missing.sum()}")
missing_report[missing_report["missing"] > 0]

Total missing cells: 3


,missing,pct
artists,1,0.0009
album_name,1,0.0009
track_name,1,0.0009


In [ ]:
df[df.isna().any(axis=1)]

,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
65900,1kR4gIb7nGxHPI3D2ifs59,NaN,NaN,NaN,0,0,False,0.501,0.583,7,-9.46,0,0.0605,0.69,0.00396,0.0747,0.734,138.391,4,k-pop


In [22]:
# Empty strings / whitespace-only values in text columns (not caught by isna)
text_cols = df.select_dtypes(include="object").columns
(df[text_cols].apply(lambda s: s.str.strip().eq("").sum()))

track_id       0
artists        0
album_name     0
track_name     0
track_genre    0
dtype: int64

## 4. Value ranges

In [23]:
num_cols = df.select_dtypes(include="number").columns
ranges = df[num_cols].agg(["min", "max", "mean", "median", "std"]).T
ranges

,min,max,mean,median,std
popularity,0.000,100.000,33.238535,35.000000,22.305078
duration_ms,0.000,5237295.000,228029.153114,212906.000000,107297.712645
danceability,0.000,0.985,0.566800,0.580000,0.173542
energy,0.000,1.000,0.641383,0.685000,0.251529
key,0.000,11.000,5.309140,5.000000,3.559987
loudness,-49.531,4.532,-8.258960,-7.004000,5.029337
mode,0.000,1.000,0.637553,1.000000,0.480709
speechiness,0.000,0.965,0.084652,0.048900,0.105732
acousticness,0.000,0.996,0.314910,0.169000,0.332523
instrumentalness,0.000,1.000,0.156050,0.000042,0.309555


In [ ]:
expected = {
    "popularity": (0, 100),
    "danceability": (0, 1), "energy": (0, 1), "speechiness": (0, 1),
    "acousticness": (0, 1), "instrumentalness": (0, 1),
    "liveness": (0, 1), "valence": (0, 1),
    "key": (-1, 11), "mode": (0, 1),
    "time_signature": (3, 7),
    "loudness": (-60, 5),
}
rows = []
for col, (lo, hi) in expected.items():
    out = ((df[col] < lo) | (df[col] > hi)).sum()
    rows.append({"column": col, "expected": f"[{lo}, {hi}]",
                 "min": df[col].min(), "max": df[col].max(), "out_of_range": out})
pd.DataFrame(rows)

,column,expected,min,max,out_of_range
0,popularity,"[0, 100]",0.000,100.000,0
1,danceability,"[0, 1]",0.000,0.985,0
2,energy,"[0, 1]",0.000,1.000,0
3,speechiness,"[0, 1]",0.000,0.965,0
4,acousticness,"[0, 1]",0.000,0.996,0
5,instrumentalness,"[0, 1]",0.000,1.000,0
6,liveness,"[0, 1]",0.000,1.000,0
7,valence,"[0, 1]",0.000,0.995,0
8,key,"[-1, 11]",0.000,11.000,0
9,mode,"[0, 1]",0.000,1.000,0


In [ ]:
print("duration_ms == 0:", (df["duration_ms"] == 0).sum())
print("Shortest track (s):", df["duration_ms"].min() / 1000)
print("Longest track (min):", round(df["duration_ms"].max() / 60000, 2))
print("tempo == 0:", (df["tempo"] == 0).sum())
print("time_signature == 0:", (df["time_signature"] == 0).sum())
print("popularity == 0:", (df["popularity"] == 0).sum())
print()
print("time_signature counts:\n", df["time_signature"].value_counts().sort_index())
print()
print("explicit counts:\n", df["explicit"].value_counts())

duration_ms == 0: 1
Shortest track (s): 0.0
Longest track (min): 87.29
tempo == 0: 157
time_signature == 0: 163
popularity == 0: 16020

time_signature counts:
 time_signature
0       163
1       973
3      9195
4    101843
5      1826
Name: count, dtype: int64

explicit counts:
 explicit
False    104253
True       9747
Name: count, dtype: int64


# Deduplication and stratified sample by genre

**Dedup rules**
1. Drop exact duplicate rows (same track listed twice in the same genre).
2. Drop the row with missing artist/album/track name.
3. Keep **one row per `track_id`**. A track listed under several genres keeps the row with its highest `popularity` (popularity is the only column that differs between copies); ties go to the first genre in file order. The genres it was dropped from are kept in `all_genres` so that information isn't lost.

In [ ]:
before = len(df)

dedup = df.drop_duplicates().copy()
print(f"Dropped {before - len(dedup):,} exact duplicate rows")

n = len(dedup)
dedup = dedup.dropna(subset=["artists", "album_name", "track_name"])
print(f"Dropped {n - len(dedup):,} row(s) with missing names")

all_genres = (dedup.groupby("track_id")["track_genre"]
                   .apply(lambda g: "|".join(sorted(set(g))))
                   .rename("all_genres"))

n = len(dedup)
dedup = (dedup.reset_index()
              .sort_values(["track_id", "popularity", "index"],
                           ascending=[True, False, True])
              .drop_duplicates(subset="track_id", keep="first")
              .sort_values("index")
              .set_index("index"))
dedup.index.name = None
dedup = dedup.join(all_genres, on="track_id")
dedup["n_genres"] = dedup["all_genres"].str.count(r"\|") + 1
print(f"Dropped {n - len(dedup):,} rows for track_ids listed under multiple genres")

assert dedup["track_id"].is_unique
print(f"\nRows: {before:,} -> {len(dedup):,}  (unique track_ids: {dedup['track_id'].nunique():,})")

Dropped 450 exact duplicate rows
Dropped 1 row(s) with missing names
Dropped 23,809 rows for track_ids listed under multiple genres

Rows: 114,000 -> 89,740  (unique track_ids: 89,740)


In [ ]:
genre_sizes = dedup["track_genre"].value_counts()
print(genre_sizes.describe().round(1))
print("\nSmallest genres after dedup:")
genre_sizes.tail(10)

count    113.0
mean     794.2
std      222.8
min       76.0
25%      708.0
50%      872.0
75%      974.0
max      999.0
Name: count, dtype: float64

Smallest genres after dedup:


track_genre
alternative    404
latino         398
rock           394
reggae         313
dubstep        268
punk           262
metal          231
house          210
indie          132
reggaeton       76
Name: count, dtype: int64

In [ ]:
lost = sorted(set(df["track_genre"]) - set(dedup["track_genre"]))
print("Genres with no rows left after dedup:", lost)

tracks_by_genre = df.groupby("track_genre")["track_id"].apply(set)
for g in lost:
    overlap = tracks_by_genre.apply(lambda s: len(s & tracks_by_genre[g])).drop(g)
    best = overlap.idxmax()
    print(f"'{g}': {len(tracks_by_genre[g])} tracks, {overlap.max()} also in '{best}'"
          f" ({overlap.max() / len(tracks_by_genre[g]):.0%}) -> its tracks were kept under '{best}'")

Genres with no rows left after dedup: ['songwriter']
'songwriter': 1000 tracks, 1000 also in 'singer-songwriter' (100%) -> its tracks were kept under 'singer-songwriter'


## Stratified sample

Draw the same number of tracks from every genre (`PER_GENRE`). If a genre has fewer tracks than that after dedup, all of its tracks are taken. Set `FRAC` instead to sample the same *fraction* of each genre (keeps the post-dedup genre proportions).

In [ ]:
PER_GENRE = 500      
FRAC = None          
SEED = 42

def stratified_sample(data, by="track_genre", n=None, frac=None, seed=SEED):
    """Sample within each genre. n: up to n rows per genre (all rows if the genre is smaller).
    frac: the same fraction of every genre."""
    if (n is None) == (frac is None):
        raise ValueError("Set exactly one of n or frac")
    if n is not None:
        return (data.sample(frac=1, random_state=seed)
                    .groupby(by, group_keys=False).head(n)
                    .sort_values(by))
    return data.groupby(by, group_keys=False).sample(frac=frac, random_state=seed)

sample = stratified_sample(dedup, n=PER_GENRE, frac=FRAC)
print(f"Sample size: {len(sample):,} rows across {sample['track_genre'].nunique()} genres")
sample.head()

Sample size: 53,942 rows across 113 genres


,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,key,...,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre,all_genres,n_genres
844,1k1jxdrfofuDZmAuLtyq6m,Frank Turner,FTHC (Deluxe),Miranda,26,240866,False,0.524,0.578,7,...,0.0267,0.000971,0.000007,0.3260,0.290,75.032,4,acoustic,acoustic,1
721,4XBcShO4h4X55x9yMmyKHL,Allman Brown,1000 Years,Sweetest Thing,54,218502,False,0.546,0.411,2,...,0.0475,0.701000,0.000097,0.0955,0.528,79.785,4,acoustic,acoustic|british,2
950,1NKiIuSbv88ecRlB6noXzz,Adam Christopher,Mood (Acoustic),Mood - Acoustic,50,161000,True,0.751,0.290,10,...,0.0456,0.846000,0.000000,0.1130,0.683,75.034,4,acoustic,acoustic,1
226,6kRZfOZSuoi0Htg6gROKDz,A Great Big World,流星花園音樂專輯,Say Something,40,229573,False,0.456,0.101,2,...,0.0380,0.876000,0.000002,0.0776,0.155,139.896,3,acoustic,acoustic|piano,2
55,6xJOhSm4SvZwzy3uhWz26O,Andrew Belle,Black Bear,The Enemy,62,286865,False,0.536,0.890,1,...,0.0375,0.078000,0.020200,0.1310,0.273,121.974,4,acoustic,acoustic|folk,2


In [ ]:
counts = sample["track_genre"].value_counts()
print("Rows per genre in sample:")
print(counts.describe().round(1))
short = genre_sizes[genre_sizes < (PER_GENRE or 0)]
print(f"\nGenres with fewer than {PER_GENRE} tracks (fully included): {len(short)}")
print(short)

print("\nDuplicate track_ids in sample:", sample["track_id"].duplicated().sum())
print("Missing values in sample:", int(sample.isna().sum().sum()))

feat = ["popularity", "danceability", "energy", "valence", "tempo"]
pd.DataFrame({"dedup_mean": dedup[feat].mean(), "sample_mean": sample[feat].mean()}).round(3)

Rows per genre in sample:
count    113.0
mean     477.4
std       73.5
min       76.0
25%      500.0
50%      500.0
75%      500.0
max      500.0
Name: count, dtype: float64

Genres with fewer than 500 tracks (fully included): 15
track_genre
r-n-b          490
j-rock         460
soul           457
techno         431
pop            416
alternative    404
latino         398
rock           394
reggae         313
dubstep        268
punk           262
metal          231
house          210
indie          132
reggaeton       76
Name: count, dtype: int64

Duplicate track_ids in sample: 0
Missing values in sample: 0


,dedup_mean,sample_mean
popularity,33.208,33.709
danceability,0.562,0.565
energy,0.634,0.636
valence,0.469,0.473
tempo,122.058,122.268


In [ ]:
sample.to_csv("spotify_stratified_sample.csv", index=False)
print("Saved spotify_stratified_sample.csv")

Saved spotify_stratified_sample.csv
